# Cross-Branch Compatibility Review

Use this notebook when two or more parallel branches may eventually be merged into the same target branch. It complements a normal Pull Request review; it does not replace it.

## 1. What is a cross-branch review?

A normal PR review asks:

> Does this branch work correctly compared with `development`?

A cross-branch compatibility review asks:

> Will these branches still work correctly when their changes are combined?

Each PR can be correct on its own but still conflict with assumptions made by another branch. This is most likely when parallel work changes shared configuration, infrastructure, data models, interfaces, or dependencies.

## 2. Example: parallel branches

```text
                    development
                   |     |      |
                   |     |      |
                 v       v        v

     feature/storage   feature/database   feature/scheduler
                   |     |      |
                   |     |      |
                    +-- eventually --+
                           |
                           v
                     development
```

For example, one branch may add object storage, another may change database configuration, and another may change a scheduled workflow. Each can work against the old `development` branch while the combined result has incompatible environment variables, paths, service names, or function calls.

## 3. When to use it

A cross-branch review is especially valuable when parallel branches change any of the following:

- `compose.yaml`, Dockerfiles, ports, volumes, or networks
- environment variables, secret names, or configuration files
- shared modules, data models, function signatures, or APIs
- database schemas, migrations, or connection settings
- dependency files such as `pyproject.toml` or lockfiles
- shared paths, storage locations, or scheduled workflows

The more central the changed component is, the more important it is to review its interaction with other branches.

## 4. Prepare the comparison

Fetch the latest remote references first:

```bash
git fetch origin
```

Write down the common baseline and the branches being compared:

```text
Baseline: origin/development
Branches:
- origin/feature/storage
- origin/feature/database
- origin/feature/scheduler
```

Then identify the purpose of each branch before comparing code. A branch name is not enough; use the PR description and actual diff to understand its responsibility.

## 5. Compare each branch with the common baseline

For each branch, inspect its commits, changed files, and full diff against the same baseline:

```bash
git log --oneline origin/development..origin/feature/storage
git diff --stat origin/development...origin/feature/storage
git diff origin/development...origin/feature/storage
```

Repeat for the other branches. Using one baseline makes it easier to see where the changes overlap and where one branch depends on an assumption made by another.

To find files changed in both branches, compare their names and then inspect the relevant sections carefully. Shared files are not automatically a problem, but they deserve extra attention.

## 6. Compatibility checklist

### Shared files and configuration

- Do branches edit the same files or the same configuration section?
- Do service names, ports, volumes, networks, and paths agree?
- Do branches use the same environment-variable names for the same purpose?

### Interfaces and data

- Did a branch change a function signature that another branch still uses?
- Do modules agree on data formats, table names, storage locations, and return values?
- Could the combined workflow duplicate processing or data?

### Dependencies and architecture

- Are dependency changes compatible when combined?
- Does the combined configuration still describe one coherent system?
- Do branches have an implicit merge order or a missing prerequisite?

## 7. Decide merge order and retesting

Sometimes one branch establishes a foundation that another branch relies on:

```text
feature/database
        |
Merge into development
        |
feature/scheduler updates from development
        |
Resolve conflicts and retest
        |
Merge feature/scheduler
```

After each merge, update the remaining branches from `development`, resolve conflicts deliberately, and retest the integration points. The final merged result still needs an end-to-end test.

## 8. Prompt example for a compatibility review

> Perform a cross-branch compatibility review for `origin/feature/storage`, `origin/feature/database`, and `origin/feature/scheduler`, using `origin/development` as the common baseline.
>
> Do not modify, merge, rebase, commit, or push anything. Determine the actual diff introduced by each branch compared with the baseline.
>
> Report: 1. Purpose of each branch, 2. Overlapping files, 3. Compatible changes, 4. Confirmed conflicts, 5. Likely integration risks, 6. Configuration or interface mismatches, 7. Recommended merge order, and 8. Tests to run after each merge.
>
> Distinguish confirmed problems from risks that need manual verification.

## Key takeaway

> A branch can pass its individual PR review and still fail when combined with another branch. Review both the code and the assumptions it makes about shared configuration, data, and interfaces.

Use `PR_Review` for the quality of a single PR, then use this notebook when parallel work creates a meaningful integration risk.